# 第26章 折线图（plot）

用折线位置和斜率表达有序时间上的趋势、转折和多序列差异。

## 学习目标

本章围绕一种明确的图表结构展开，先看最小可用示例，再加入分组、注释或交互细节。学习重点不是“把图画出来”，而是让图表服务于一个可回答的问题。


## 适用场景

X轴具有自然顺序，重点是观察连续变化、增长速度或周期。

## 数据结构

一列有序时间或阶段，一列或多列同单位指标；缺失时间点需要显式处理。

## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 marker='o' 改为 marker='s'，观察标记形状变化
2. 调整 linewidth 参数（如 0.5 或 3.5），说明线条粗细对可读性的影响
3. 修改 linestyle 为 '--'，对比虚线与实线的视觉效果


## 0. 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


transactions = pd.read_csv(f"{base_url}/datasets/uci_online_retail_200k.csv", parse_dates=["InvoiceDate"])
transactions["amount"] = transactions["Quantity"] * transactions["UnitPrice"]
transactions["month"] = transactions["InvoiceDate"].dt.to_period("M").astype("string")
completed = transactions.query("Quantity > 0 and UnitPrice > 0")
monthly_summary = completed.groupby("month").agg(sales=("amount", "sum"), orders=("InvoiceNo", "nunique"))
months = monthly_summary.index.to_numpy()
sales = (monthly_summary["sales"] / 10_000).to_numpy()
orders = monthly_summary["orders"].to_numpy()
profit = sales * 0.18
top_countries = completed.groupby("Country")["amount"].sum().nlargest(4).index
country_rows = transactions[transactions["Country"].isin(top_countries)].copy()
country_rows["flow"] = np.where(country_rows["Quantity"] > 0, "销售", "退货")
country_rows["amount_abs"] = country_rows["amount"].abs()
regional_summary = country_rows.pivot_table(index="Country", columns="flow", values="amount_abs", aggfunc="sum", fill_value=0) / 10_000
regions = regional_summary.index.to_numpy()
online = regional_summary.get("销售", pd.Series(0, index=regional_summary.index)).to_numpy()
offline = regional_summary.get("退货", pd.Series(0, index=regional_summary.index)).to_numpy()
samples = completed["amount"].sample(2_000, random_state=25).to_numpy()
print(f"UCI Online Retail：{len(transactions):,} 行；图表使用聚合结果与固定样本")


## 1. 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales, marker="o", linewidth=2.2, color="#1a73e8")
ax.set(title="上半年销售额趋势", xlabel="月份", ylabel="销售额（万元）")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.2)
fig.tight_layout()
plt.show()


## 2. 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
sales_index = sales / sales[0] * 100
profit_index = profit / profit[0] * 100
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, sales_index, marker="o", label="销售额指数")
ax.plot(months, profit_index, marker="s", label="利润指数")
ax.axhline(100, color="#9aa0a6", linestyle="--", linewidth=1)
ax.set(title="利润增长快于销售额", ylabel="指数（1月=100）")
ax.legend(frameon=False)
fig.tight_layout()
plt.show()


## 3. 参数说明

- marker：观测点
- linestyle：线型
- linewidth：线宽
- label：序列名称


## 4. 结果解读

先读总体方向，再找峰谷、转折和序列间差距；不能把连接线误解为未观测区间的真实数据。


## 常见误区

- 用折线连接无顺序类别
- 多条线颜色过近
- 时间缺失却直接连接


## 综合练习

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


In [ ]:
growth = np.diff(orders)
fastest = int(growth.argmax()) + 1
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(months, orders, marker="o", color="#188038")
ax.scatter(months[fastest], orders[fastest], s=90, color="#d93025", zorder=3)
ax.annotate(f"增加 {growth[fastest - 1]} 单", (months[fastest], orders[fastest]), xytext=(-45, 25), textcoords="offset points", arrowprops={"arrowstyle": "->"})
ax.set(title="订单量趋势及最大增量", ylabel="订单数")
fig.tight_layout()
plt.show()


## 本章小结

用折线位置和斜率表达有序时间上的趋势、转折和多序列差异。


### 你已经掌握

- 判断折线图（plot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 图表选择速查

| 选择要点 | 本章说明 |
| --- | --- |
| 适用场景 | X轴具有自然顺序，重点是观察连续变化、增长速度或周期。 |
| 数据结构 | 一列有序时间或阶段，一列或多列同单位指标；缺失时间点需要显式处理。 |
| 结果解读 | 先读总体方向，再找峰谷、转折和序列间差距；不能把连接线误解为未观测区间的真实数据。 |


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `marker` | 观测点 |
| `linestyle` | 线型 |
| `linewidth` | 线宽 |
| `label` | 序列名称 |


### 需要注意

- 用折线连接无顺序类别
- 多条线颜色过近
- 时间缺失却直接连接


### 完成检查

- [ ] 能判断什么问题适合使用折线图（plot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
